# 🔬 Agentic Research Analyst — Exploration Notebook

Interactive walkthrough of the agent pipeline:
1. Run a single research brief
2. Inspect the tool-use trace
3. Visualise cost & latency breakdown
4. Compare memo quality across domains

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

# Verify keys are set
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"
assert os.getenv("TAVILY_API_KEY"), "Set TAVILY_API_KEY in .env"
print("✅ Environment ready")

## 1. Run a Single Brief

In [ ]:
from src.agents.research_agent import ResearchAgent

agent = ResearchAgent()

brief = """
Compare the architectural trade-offs between Transformer and
State Space Model (SSM) approaches for long-context language
modelling. Cover: attention complexity, memory footprint,
benchmark performance, and production readiness.
"""

result = agent.run(brief)
print(f"Status:       {result['status']}")
print(f"Tool calls:   {len(result['tool_trace'])}")
print(f"Cost:         ${result['metadata']['total_cost_usd']:.4f}")
print(f"Latency:      {result['metadata']['total_latency_s']:.1f}s")

## 2. Inspect the Memo

In [ ]:
from IPython.display import Markdown

Markdown(result["memo"].to_markdown())

## 3. Tool-Use Trace

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(result["tool_trace"])
trace_df[["tool", "latency_ms", "tokens_used", "cost_usd"]]

## 4. Cost & Latency Breakdown

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cost by tool
cost_by_tool = trace_df.groupby("tool")["cost_usd"].sum()
cost_by_tool.plot.bar(ax=axes[0], color="#6366f1")
axes[0].set_title("Cost by Tool ($)")
axes[0].set_ylabel("USD")

# Latency by tool
lat_by_tool = trace_df.groupby("tool")["latency_ms"].sum()
lat_by_tool.plot.bar(ax=axes[1], color="#10b981")
axes[1].set_title("Latency by Tool (ms)")
axes[1].set_ylabel("ms")

plt.tight_layout()
plt.show()

## 5. Run Evaluation Suite

In [ ]:
from src.evaluation.run import run_evaluation
from src.evaluation.metrics import compute_aggregate_metrics

results = run_evaluation(suite="smoke")
metrics = compute_aggregate_metrics(results)

print(f"Task Completion:  {metrics['task_completion_pct']:.1f}%")
print(f"Mean Quality:     {metrics['mean_quality_score']:.2f}/10")
print(f"Avg Cost/Brief:   ${metrics['avg_cost_per_brief']:.4f}")
print(f"Avg Tool Calls:   {metrics['avg_tool_calls']:.1f}")

## 6. Domain-Level Comparison

In [ ]:
domain_df = pd.DataFrame(metrics["per_domain"])

fig, ax = plt.subplots(figsize=(8, 4))
domain_df.plot.bar(
    x="domain",
    y=["completeness", "citation", "coherence"],
    ax=ax,
    color=["#6366f1", "#f59e0b", "#10b981"],
)
ax.set_title("Quality Scores by Domain")
ax.set_ylabel("Score (1-10)")
ax.set_ylim(0, 10)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()